In [1]:
#!/usr/bin/env python
# coding: utf-8

# # Import libraries

# In[1]:


# Importing libraries
from google.cloud import bigquery
import pandas as pd
from pandas_gbq import to_gbq
import os
import subprocess

print('Libraries imported successfully')


# In[2]:


# Setting the environment variable for Google Cloud credentials
# Placing the path in which the .json file is located.

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = r"C:\Users\roxan\AppData\Roaming\gcloud\application_default_credentials.json"
# In[3]:


# Setting the Google Cloud project ID and BigQuery dataset details

project_id = 'project-401f4646-3663-4125-aaa' # project id
dataset_id = 'reporting_db' # schema name: staging_db, reporting_db etc.
table_id = 'rep_rentals_per_customer_and_period' # table name: stg_customer, stg_city etc.

# # SQL Query

# In[4]:


# Creating a BigQuery client
client = bigquery.Client(project=project_id)

# Defining the SQL query here
query = """ 
   
   with cte_rentals as (

      select * 
      from `project-401f4646-3663-4125-aaa.staging_db.stg_rental`

  )

  , cte_reporting_dates as (

      select * 
      from `project-401f4646-3663-4125-aaa.reporting_db.reporting_periods_table`
      where reporting_period in ('Day','Month','Year')

  )

  ,  cte_customers as (
     select *
     from `project-401f4646-3663-4125-aaa.staging_db.stg_customer`
  )

  ,  cte_rentals_per_period as (

      select
          'Day' as reporting_period,
          CAST(date_trunc(rent.rental_rental_date, day) AS DATE) as reporting_date, -- Added CAST
          cust.customer_id,
          count(*) as total_rentals
      from cte_rentals as rent left join cte_customers as cust 
            on rent.rental_customer_id = cust.customer_id
      group by reporting_period,reporting_date,cust.customer_id
  
      union all

      select
          'Month' as reporting_period,
          CAST(date_trunc(rent.rental_rental_date, month) AS DATE) as reporting_date, -- Added CAST
          cust.customer_id,
          count(*) as total_rentals
      from cte_rentals as rent left join cte_customers as cust 
            on rent.rental_customer_id = cust.customer_id
      group by reporting_period,reporting_date,cust.customer_id

      union all

      select
          'Year' as reporting_period,
          CAST(date_trunc(rent.rental_rental_date, year) AS DATE) as reporting_date, -- Added CAST
          cust.customer_id,
          count(*) as total_rentals
      from cte_rentals as rent left join cte_customers as cust 
            on rent.rental_customer_id = cust.customer_id
      group by reporting_period,reporting_date,cust.customer_id

  )
  
 -- All above combined with all dates master date table
 ,  cte_final as (

      select 
          cte_reporting_dates.reporting_period,
          cte_reporting_dates.reporting_date,
          cte_rentals_per_period.customer_id,
          cte_rentals_per_period.total_rentals as total_rentals
      from cte_reporting_dates inner join cte_rentals_per_period
        on cte_reporting_dates.reporting_period=cte_rentals_per_period.reporting_period 
        and cte_reporting_dates.reporting_date=cte_rentals_per_period.reporting_date
      where cte_reporting_dates.reporting_period = 'Day'

      union all

      select 
          cte_reporting_dates.reporting_period,
          cte_reporting_dates.reporting_date,
          cte_rentals_per_period.customer_id,
          cte_rentals_per_period.total_rentals as total_rentals
      from cte_reporting_dates inner join cte_rentals_per_period
        on cte_reporting_dates.reporting_period=cte_rentals_per_period.reporting_period 
        and cte_reporting_dates.reporting_date=cte_rentals_per_period.reporting_date 
      where cte_reporting_dates.reporting_period = 'Month'

      union all

      select 
          cte_reporting_dates.reporting_period,
          cte_reporting_dates.reporting_date,
          cte_rentals_per_period.customer_id,
          cte_rentals_per_period.total_rentals as total_rentals
      from cte_reporting_dates inner join cte_rentals_per_period 
        on cte_reporting_dates.reporting_period=cte_rentals_per_period.reporting_period 
        and cte_reporting_dates.reporting_date=cte_rentals_per_period.reporting_date 
      where cte_reporting_dates.reporting_period = 'Year'

 )

  select * from cte_final;
  
"""

# Executing the query and storing the result in a dataframe
df = client.query(query).to_dataframe()

# # Writing to BigQuery

# In[5]:

# Defining the full table ID
full_table_id = f"{project_id}.{dataset_id}.{table_id}"

# Exploring some records
display(df.head())

# Defining table schema
schema = [
    bigquery.SchemaField('reporting_period', 'STRING'),
    bigquery.SchemaField('reporting_date', 'DATE'),
    bigquery.SchemaField('customer_id', 'INTEGER'),
    bigquery.SchemaField('total_rentals', 'INTEGER')
]
# In[6]:

# Creating a BigQuery client
client = bigquery.Client(project=project_id)

# Configuring the load job to always overwrite if the table exists, or creating if it doesn't
job_config = bigquery.LoadJobConfig(
    schema=schema,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE 
)

print(f"Loading data into {full_table_id}...")

# Running the job
job = client.load_table_from_dataframe(df, full_table_id, job_config=job_config)
job.result()  # Wait for the job to complete

print(f"Data successfully loaded to {full_table_id}.")


# In[7]:

# Safely running terminal commands 
try:
    subprocess.run(['python', '-m', 'pip', 'install', 'nbconvert', '-U'], check=True)
    subprocess.run(['python', '-m', 'jupyter', 'nbconvert', 'rep_rentals_per_customer_and_period.ipynb', '--to', 'python', '--output-dir=..'], check=True)
    print("Notebook successfully converted.")
except Exception as e:
    print(f"Notebook conversion skipped or failed: {e}")

Libraries imported successfully


C:\Users\roxan\PycharmProjects\Pagila Analytics Pipeline\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,reporting_period,reporting_date,customer_id,total_rentals
0,Day,2022-05-31,5,2
1,Day,2022-05-31,7,1
2,Day,2022-05-31,9,1
3,Day,2022-05-31,12,1
4,Day,2022-05-31,16,1


Loading data into project-401f4646-3663-4125-aaa.reporting_db.rep_rentals_per_customer_and_period...
Data successfully loaded to project-401f4646-3663-4125-aaa.reporting_db.rep_rentals_per_customer_and_period.
Notebook successfully converted.
